# UniChart — Custom `marker_map`

Demonstrates and verifies the user-customizable **`marker_map`** — the ordered
list of marker symbols assigned to datasets by index (the marker analogue of
`color_map`).

Covered:

- **Default** map = the built-in marker symbol set.
- **Cyclic indexing**: `marker_map[3]` works on a 2-marker map (it wraps).
- **Custom map set before load** → datasets cycle through your markers.
- **Expectation management**: reassigning `marker_map` on an *existing* notebook
  does **not** retroactively restyle loaded datasets; `reset_format()` applies it.
- **Integer `marker()` lookups** resolve through the custom map.
- **Empty-map guard** falls back to the default marker set.

**Interactive:** every `UnichartNotebook` built here is left in a top-level
variable (`uc_default`, `uc_custom`, `uc_existing`, `uc_int`, `uc_empty`) so you
can keep experimenting after running — e.g. `uc_custom.marker_map = ['o','x']`
then `uc_custom.reset_format(); uc_custom.plot('x','y')`. The last section lists
them.

Each section makes hard `assert`s (the real test) and renders a plot. The summary
prints **ALL MARKER_MAP TESTS PASSED** if every assertion held.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook, MARKER_MAP_MPL_TO_PLOTLY

DEFAULT_MARKERS = list(MARKER_MAP_MPL_TO_PLOTLY.keys())
# A short 3-marker custom map, so cycling is visible across 6 datasets.
CUSTOM = ['o', 's', '^']   # circle / square / triangle

PASSED = 0
def check(cond, msg):
    global PASSED
    assert cond, 'FAILED: ' + msg
    PASSED += 1
    print('PASS:', msg)

def build(nb, n=6, marker_map=None):
    """Populate an EXISTING notebook with n datasets (markers visible as scatter).
    Pass marker_map to set it BEFORE loading. Returns the same object."""
    if marker_map is not None:
        nb.marker_map = list(marker_map)
    x = np.linspace(0, 10, 18)
    for i in range(n):
        nb.load_df(pd.DataFrame({'x': x, 'y': x + i * 3}), title=f'S{i}')
    nb.markersize('all', 12)   # bigger markers so the symbol is easy to read
    return nb

def expected(mmap, n):
    return [mmap[i % len(mmap)] for i in range(n)]

print('default marker_map (first 5):', UnichartNotebook().marker_map[:5])
print('CUSTOM map:', CUSTOM)

UniChart Notebook Environment Initialized.
default marker_map (first 5): ['o', 's', 'D', 'd', 'v']
CUSTOM map: ['o', 's', '^']


## A. Default map = built-in marker set

Out of the box, datasets get distinct markers from the default symbol sequence.

In [2]:
uc_default = build(UnichartNotebook(), 6)   # exposed top-level object
markers = [d.marker for d in uc_default.sets]
print('dataset markers:', markers)
check(markers == expected(DEFAULT_MARKERS, 6), "default datasets use the built-in marker set by index")
uc_default.plot('x', 'y', suptitle='A. Default marker_map').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
dataset markers: ['o', 's', 'D', 'd', 'v', '^']
PASS: default datasets use the built-in marker set by index


## B. Cyclic indexing — `marker_map[3]` on a short map

`marker_map` is a `CyclicList`: integer indexing wraps, so a short map still
answers any index. Slicing stays normal.

In [3]:
uc_custom = UnichartNotebook()             # exposed top-level object
uc_custom.marker_map = ['o', 's']          # length 2
print('marker_map[0..4]:', [uc_custom.marker_map[i] for i in range(5)])
check([uc_custom.marker_map[i] for i in range(5)] == ['o', 's', 'o', 's', 'o'],
      "integer indexing cycles through a 2-marker map")
check(uc_custom.marker_map[3] == 's', "marker_map[3] on a length-2 map wraps to 's'")
check(uc_custom.marker_map[:1] == ['o'] and uc_custom.marker_map[0:5] == ['o', 's'],
      "slicing is NOT cyclic (normal list behavior)")

UniChart Notebook Environment Initialized.
marker_map[0..4]: ['o', 's', 'o', 's', 'o']
PASS: integer indexing cycles through a 2-marker map
PASS: marker_map[3] on a length-2 map wraps to 's'
PASS: slicing is NOT cyclic (normal list behavior)


## C. Custom map set before loading

With a 3-marker map and 6 datasets the symbols repeat: circle, square, triangle,
circle, square, triangle.

In [4]:
build(uc_custom, 6, marker_map=CUSTOM)      # reuse the exposed object from B
markers = [d.marker for d in uc_custom.sets]
print('dataset markers:', markers)
check(markers == expected(CUSTOM, 6), "custom map set before load cycles across datasets")
check(markers == ['o', 's', '^', 'o', 's', '^'], "6 datasets over a 3-marker map repeat o/s/^")
uc_custom.plot('x', 'y', suptitle='C. Custom marker_map (o / s / ^, cycling)').show()

Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
dataset markers: ['o', 's', '^', 'o', 's', '^']
PASS: custom map set before load cycles across datasets
PASS: 6 datasets over a 3-marker map repeat o/s/^


## D. Expectation management — applying a new map to an existing notebook

`marker_map` is read when a dataset is created. Reassigning it later does **not**
retroactively restyle existing datasets — `reset_format()` (or loading new data)
applies the new map.

In [5]:
uc_existing = build(UnichartNotebook(), 6)   # exposed; loaded with the default map
uc_existing.plot('x', 'y', suptitle='D1. Before: default markers').show()

uc_existing.marker_map = CUSTOM              # change the map AFTER loading
check([d.marker for d in uc_existing.sets] == expected(DEFAULT_MARKERS, 6),
      "reassigning marker_map does NOT retroactively restyle existing datasets")

uc_existing.reset_format()                   # now apply it
check([d.marker for d in uc_existing.sets] == expected(CUSTOM, 6),
      "reset_format() re-applies the (custom) marker_map to existing datasets")
uc_existing.markersize('all', 12)
uc_existing.plot('x', 'y', suptitle='D2. After reset_format(): custom markers').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5


PASS: reassigning marker_map does NOT retroactively restyle existing datasets
Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.
PASS: reset_format() re-applies the (custom) marker_map to existing datasets


## E. Integer `marker()` lookups use the custom map

`marker(target, <int>)` resolves the integer through `marker_map`, so a dataset
can borrow another index's mapped marker.

In [6]:
uc_int = build(UnichartNotebook(), 6, marker_map=CUSTOM)   # exposed
uc_int.marker(0, 4)   # index 4 -> CUSTOM[4 % 3] = CUSTOM[1] = 's'
print('set 0 marker after marker(0, 4):', uc_int.sets[0].marker)
check(uc_int.sets[0].marker == CUSTOM[4 % len(CUSTOM)], "int marker() resolves through the custom map")
check(uc_int.sets[0].marker == 's', "marker(0, 4) -> square (CUSTOM[1])")

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
Loaded Set 3: S3
Loaded Set 4: S4
Loaded Set 5: S5
set 0 marker after marker(0, 4): s
PASS: int marker() resolves through the custom map
PASS: marker(0, 4) -> square (CUSTOM[1])


## F. Empty-map guard

If `marker_map` is set to an empty list, lookups safely fall back to the default
marker set instead of raising.

In [7]:
uc_empty = build(UnichartNotebook(), 3)   # exposed
uc_empty.marker_map = []
check(uc_empty._marker_at(1) == DEFAULT_MARKERS[1], "_marker_at falls back to the default set when empty")
uc_empty.load_df(pd.DataFrame({'x': [1], 'y': [1]}), title='extra')
check(uc_empty.sets[-1].marker == DEFAULT_MARKERS[3], "new dataset under empty map falls back to default[index]")
print('no crash; fallback marker:', uc_empty.sets[-1].marker)

UniChart Notebook Environment Initialized.
Loaded Set 0: S0
Loaded Set 1: S1
Loaded Set 2: S2
PASS: _marker_at falls back to the default set when empty
Loaded Set 3: extra
PASS: new dataset under empty map falls back to default[index]
no crash; fallback marker: d


## Interact with the exposed objects

These `UnichartNotebook` instances remain in the namespace — inspect or restyle
them freely.

In [8]:
for name in ['uc_default', 'uc_custom', 'uc_existing', 'uc_int', 'uc_empty']:
    nb = globals()[name]
    print(f"{name:12s} | {len(nb.sets)} sets | marker_map={list(nb.marker_map)}")

# Example you can run/edit live:
#   uc_custom.marker_map = ['x', 'star', 'diamond']
#   uc_custom.reset_format()
#   uc_custom.markersize('all', 14)
#   uc_custom.plot('x', 'y')
print('\nTip: e.g.  uc_custom.marker_map = [\'x\',\'star\']; uc_custom.reset_format(); uc_custom.plot(\'x\',\'y\')')

uc_default   | 6 sets | marker_map=['o', 's', 'D', 'd', 'v', '^', '<', '>', 'p', '*', 'h', 'H', 'x', 'X', '+', '|', '_', '.']
uc_custom    | 6 sets | marker_map=['o', 's', '^']
uc_existing  | 6 sets | marker_map=['o', 's', '^']
uc_int       | 6 sets | marker_map=['o', 's', '^']
uc_empty     | 4 sets | marker_map=[]

Tip: e.g.  uc_custom.marker_map = ['x','star']; uc_custom.reset_format(); uc_custom.plot('x','y')


## Summary

In [9]:
print(f'{PASSED} assertions passed.')
print('ALL MARKER_MAP TESTS PASSED' if PASSED >= 10 else 'CHECK COUNT UNEXPECTED')

12 assertions passed.
ALL MARKER_MAP TESTS PASSED
